# [3.6] Thumbs-up / Thumbs-down 학습 노트북 (Jetson Orin Nano)
환경 확인 → 학습 → 평가 → ONNX · TensorRT 내보내기. 교안 3.6.2 실습 과제 ①~⑤.

## ① 환경 확인

In [ ]:
import torch, ultralytics
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))   # Orin
print("ultralytics", ultralytics.__version__)

## ② 데이터셋 확인
`python download_dataset.py`로 받은 `dataset/data.yaml`을 확인합니다 (nc=2, Thumbs-down=0, Thumbs-up=1).

In [ ]:
print(open("dataset/data.yaml", encoding="utf-8").read())

## ③ 학습

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
model.train(
    data="dataset/data.yaml",
    epochs=20, imgsz=640, batch=8,   # Orin Nano: CPU/GPU가 8GB 공유 → batch=8 (16이면 OOM 위험)
    device=0, workers=2,
    amp=True,                        # Ampere 텐서코어로 FP16 혼합 정밀도 학습
    name="thumb",
    exist_ok=True,                   # 다시 학습해도 thumb2, thumb3… 가 아니라 thumb 폴더에 덮어씀
)                                    # → runs/detect/thumb/weights/best.pt

## ④ 평가 · 결과 그래프 확인

In [ ]:
from IPython.display import Image, display

best = YOLO("runs/detect/thumb/weights/best.pt")
metrics = best.val(data="dataset/data.yaml", imgsz=640, batch=8, device=0)
print(f"mAP50 = {metrics.box.map50:.3f} | mAP50-95 = {metrics.box.map:.3f}")

display(Image("runs/detect/thumb/results.png", width=900))
display(Image("runs/detect/thumb/confusion_matrix_normalized.png", width=600))

## ⑤ ONNX · TensorRT 내보내기
변환 파일은 `runs/detect/thumb/weights/`에 생기므로, `predict.py`가 바로 쓰도록 이 폴더로 복사합니다. TensorRT 변환은 **10분 이상** 걸립니다.

In [ ]:
import shutil

best = YOLO("runs/detect/thumb/weights/best.pt")
onnx_path = best.export(format="onnx")                              # → weights/best.onnx
engine_path = best.export(format="engine", half=True, workspace=2)  # → weights/best.engine (TensorRT FP16)

shutil.copy(onnx_path, "best.onnx")       # predict.py 가 이 폴더에서 먼저 찾음
shutil.copy(engine_path, "best.engine")
print("복사 완료: best.onnx, best.engine")